# 9.4 Causality: seeing, doing and imagining: practice notebook

Tell correlation from cause. Use the do operator and graph surgery for actions, the back door adjustment to undo confounding (and explain Simpson’s paradox), and structural causal models for counterfactual questions.

How to use it: run the cells from top to bottom (Shift + Enter). First read and run the examples. Then try each task in its starter cell and run the check cell under it. The full solutions are at the end, so try on your own first.

Everything here is plain Python 3. No extra packages are needed.

## Setup

In [ ]:
# A small helper. check(tests) runs each test of a task and tells you which ones pass.
# Each test is a pair: the call to make, and the answer we expect.
import json

def check(tests):
    passed = 0
    for call, expected in tests:
        try:
            got = eval(call)
        except Exception as error:
            got = "error: " + str(error)
        same = json.dumps(got, sort_keys=True, default=repr) == json.dumps(expected, sort_keys=True, default=repr)
        if same:
            passed = passed + 1
            print("PASS ", call)
        else:
            print("FAIL ", call, " expected", expected, " but got", got)
    if passed == len(tests):
        print("All", len(tests), "tests pass. Well done!")
    else:
        print(passed, "of", len(tests), "tests pass. Keep going, or look at the solution below.")

## Examples from the chapter

### Seeing versus doing in the sprinkler network

In [ ]:
# Seeing versus doing in the sprinkler network: Cloudy -> Sprinkler, Cloudy -> Rain, both -> WetGrass.
p_c = 0.5
p_s = {True: 0.1, False: 0.5}      # P(Sprinkler = true | Cloudy)
p_r = {True: 0.8, False: 0.2}      # P(Rain = true | Cloudy)
p_w = {(True, True): 0.99, (True, False): 0.90, (False, True): 0.90, (False, False): 0.0}

# Seeing: P(Rain | Sprinkler = true). The sprinkler being on tells us something about Cloudy.
top, bottom = 0, 0
for c in [True, False]:
    pc = p_c if c else 1 - p_c
    top = top + pc * p_s[c] * p_r[c]
    bottom = bottom + pc * p_s[c]
print("P(Rain | Sprinkler = true)     =", round(top / bottom, 3))

# Doing: do(Sprinkler = true) cuts Cloudy -> Sprinkler, so P(s | c) leaves the product.
rain = 0
for c in [True, False]:
    pc = p_c if c else 1 - p_c
    rain = rain + pc * p_r[c]
print("P(Rain | do(Sprinkler = true)) =", round(rain, 3))

# Doing, then looking at an effect: P(WetGrass | do(Sprinkler = true)).
wet = 0
for c in [True, False]:
    pc = p_c if c else 1 - p_c
    for r in [True, False]:
        pr = p_r[c] if r else 1 - p_r[c]
        wet = wet + pc * pr * p_w[(True, r)]
print("P(WetGrass | do(Sprinkler = true)) =", round(wet, 3))

### Simpson’s paradox and the back door adjustment

In [ ]:
# Kidney stones (Charig and others, 1986): (stone size, treatment, recovered, total).
data = [("small", "A", 81, 87), ("small", "B", 234, 270),
        ("large", "A", 192, 263), ("large", "B", 55, 80)]
everyone = sum(n for z, t, k, n in data)

def observed(t):                       # seeing: P(recovery | T = t)
    k = sum(k for z, tt, k, n in data if tt == t)
    n = sum(n for z, tt, k, n in data if tt == t)
    return k / n

def adjusted(t):                       # doing: P(recovery | do(T = t)) = sum over z of P(r | t, z) P(z)
    total = 0
    for z in ["small", "large"]:
        pz = sum(n for zz, tt, k, n in data if zz == z) / everyone
        for zz, tt, k, n in data:
            if zz == z and tt == t:
                total = total + (k / n) * pz
    return total

for t in ["A", "B"]:
    print(t, " seeing:", round(observed(t), 3), " doing:", round(adjusted(t), 3))
print("causal effect of A over B:", round(adjusted("A") - adjusted("B"), 3))

## Your turn

### Task 1: The observed recovery rate

Write `observed(data, t)` that returns recovered ÷ total over all rows with treatment t.

Hint: Add recovered and totals for rows with this treatment, then divide.

In [ ]:
DATA = [("small", "A", 81, 87), ("small", "B", 234, 270),
        ("large", "A", 192, 263), ("large", "B", 55, 80)]   # (group, treatment, recovered, total)
EVEN = [("small", "A", 90, 100), ("small", "B", 80, 100),
        ("large", "A", 60, 100), ("large", "B", 50, 100)]

def observed(data, t):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("round(observed(DATA, 'A'), 3)", 0.78), ("round(observed(DATA, 'B'), 3)", 0.826), ("round(observed(EVEN, 'A'), 3)", 0.75)])

### Task 2: The back door adjustment

Write `adjusted(data, t)` = Σ over groups z of P(recovery | t, z) × P(z), where P(z) is the share of all patients in group z.

Hint: First find the share of each group among all patients. Then add rate × share for the rows with treatment t.

In [ ]:
DATA = [("small", "A", 81, 87), ("small", "B", 234, 270),
        ("large", "A", 192, 263), ("large", "B", 55, 80)]   # (group, treatment, recovered, total)
EVEN = [("small", "A", 90, 100), ("small", "B", 80, 100),
        ("large", "A", 60, 100), ("large", "B", 50, 100)]

def adjusted(data, t):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("round(adjusted(DATA, 'A'), 3)", 0.833), ("round(adjusted(DATA, 'B'), 3)", 0.779), ("round(adjusted(EVEN, 'A'), 3)", 0.75)])

### Task 3: A do query by graph surgery

For the sprinkler network, write `wet_do(s)` = P(WetGrass = True | do(Sprinkler = s)) and `wet_see(s)` = P(WetGrass = True | Sprinkler = s). Sum over Cloudy and Rain. For do, leave P(s | c) out of the product.

Hint: Both are sums over c and r. Seeing keeps the factor P(s | c) and divides by P(s). Doing drops it.

In [ ]:
P_C = 0.5
P_S = {True: 0.1, False: 0.5}
P_R = {True: 0.8, False: 0.2}
P_W = {(True, True): 0.99, (True, False): 0.90, (False, True): 0.90, (False, False): 0.0}

def wet_do(s):
    # your code here
    pass

def wet_see(s):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([('round(wet_do(True), 3)', 0.945), ('round(wet_see(True), 3)', 0.927), ('round(wet_do(False), 3)', 0.45), ('round(wet_see(False), 3)', 0.527)])

## Solutions

Try the tasks yourself first. Running a solution cell replaces your own function with the model one.

### Solution to task 1: The observed recovery rate

Overall B looks better: 0.826 against 0.78.

In [ ]:
DATA = [("small", "A", 81, 87), ("small", "B", 234, 270),
        ("large", "A", 192, 263), ("large", "B", 55, 80)]   # (group, treatment, recovered, total)
EVEN = [("small", "A", 90, 100), ("small", "B", 80, 100),
        ("large", "A", 60, 100), ("large", "B", 50, 100)]

def observed(data, t):
    k = 0
    n = 0
    for group, treat, rec, tot in data:
        if treat == t:
            k = k + rec
            n = n + tot
    return k / n

In [ ]:
check([("round(observed(DATA, 'A'), 3)", 0.78), ("round(observed(DATA, 'B'), 3)", 0.826), ("round(observed(EVEN, 'A'), 3)", 0.75)])

### Solution to task 2: The back door adjustment

Test 3: with even groups, seeing and doing give the same 0.75.

In [ ]:
DATA = [("small", "A", 81, 87), ("small", "B", 234, 270),
        ("large", "A", 192, 263), ("large", "B", 55, 80)]   # (group, treatment, recovered, total)
EVEN = [("small", "A", 90, 100), ("small", "B", 80, 100),
        ("large", "A", 60, 100), ("large", "B", 50, 100)]

def adjusted(data, t):
    everyone = 0
    for group, treat, rec, tot in data:
        everyone = everyone + tot
    groups = []
    for group, treat, rec, tot in data:
        if group not in groups:
            groups.append(group)
    result = 0
    for z in groups:
        size = 0
        for group, treat, rec, tot in data:
            if group == z:
                size = size + tot
        for group, treat, rec, tot in data:
            if group == z and treat == t:
                result = result + (rec / tot) * (size / everyone)
    return result

In [ ]:
check([("round(adjusted(DATA, 'A'), 3)", 0.833), ("round(adjusted(DATA, 'B'), 3)", 0.779), ("round(adjusted(EVEN, 'A'), 3)", 0.75)])

### Solution to task 3: A do query by graph surgery

Turning the sprinkler off ourselves leaves rain as the only cause: 0.5 × 0.9 = 0.45. Seeing it off suggests clouds, so rain is more likely (0.586), and the grass is wet with chance 0.586 × 0.9 ≈ 0.527.

In [ ]:
P_C = 0.5
P_S = {True: 0.1, False: 0.5}
P_R = {True: 0.8, False: 0.2}
P_W = {(True, True): 0.99, (True, False): 0.90, (False, True): 0.90, (False, False): 0.0}

def wet_do(s):
    total = 0
    for c in [True, False]:
        pc = P_C if c else 1 - P_C
        for r in [True, False]:
            pr = P_R[c] if r else 1 - P_R[c]
            total = total + pc * pr * P_W[(s, r)]
    return total

def wet_see(s):
    top = 0
    bottom = 0
    for c in [True, False]:
        pc = P_C if c else 1 - P_C
        ps = P_S[c] if s else 1 - P_S[c]
        bottom = bottom + pc * ps
        for r in [True, False]:
            pr = P_R[c] if r else 1 - P_R[c]
            top = top + pc * ps * pr * P_W[(s, r)]
    return top / bottom

In [ ]:
check([('round(wet_do(True), 3)', 0.945), ('round(wet_see(True), 3)', 0.927), ('round(wet_do(False), 3)', 0.45), ('round(wet_see(False), 3)', 0.527)])